In [1]:
# =====================================================================
#  Social Media Privacy Risk Assessment Framework  |  single-cell Colab build
#  Defensive, educational. Synthetic / self-reported data only. No scraping.
#  Run this ONE cell: it builds data, engine, DB, API, dashboard, tests, and a public URL.
# =====================================================================
import os, re, json, time, uuid, html, socket, sqlite3, logging, threading, subprocess, datetime as dt
import numpy as np, pandas as pd
from flask import Flask, request, jsonify, Response

ROOT = "/content/Social-Media-Privacy-Risk-Assessment" if os.path.isdir("/content") else os.path.abspath("Social-Media-Privacy-Risk-Assessment")
for d in ("data", "reports", "tests", "docs", "screenshots"): os.makedirs(f"{ROOT}/{d}", exist_ok=True)
DB = f"{ROOT}/privacy_assessments.db"
DISCLAIMER = ("Educational risk framework: it is not a guarantee that an account will or will not be compromised. "
              "Weights and thresholds are educational assumptions and must be validated before any professional risk decision.")

# ---------------------------------------------------------------- 1. QUESTIONNAIRE (42 questions)
CATS = {"profile": "Profile Visibility", "personal": "Personal Information", "location": "Location Privacy",
        "content": "Posts & Content", "connections": "Connections", "tagging": "Tagging & Mentions",
        "account": "Account Security", "apps": "Third-Party Apps", "social": "Social Engineering", "footprint": "Digital Footprint"}
WEIGHTS = {"profile": .10, "personal": .15, "location": .15, "content": .10, "connections": .10,
           "tagging": .05, "account": .15, "apps": .05, "social": .10, "footprint": .05}   # configurable, sums to 1.0
# key|category|kind|weight(1-3)|question|finding|recommendation
# kind: vis = PUBLIC/FRIENDS/PRIVATE, bad = YES is risky, good = YES is protective
RAW = """profile_public|profile|vis|3|Who can see your profile by default?|Profile is publicly visible|Set the profile audience to Friends or Private where possible.
search_indexed|profile|bad|2|Can search engines show your profile in their results?|Profile is indexable by search engines|Turn off search-engine indexing of your profile.
friends_list_public|profile|bad|1|Is your friends/followers list visible to everyone?|Friends list is public|Restrict who can see your friends/followers list.
discoverable|profile|bad|2|Can strangers find you using your phone number or email?|Profile is discoverable via contact details|Limit who can look you up by phone or email.
phone_public|personal|bad|3|Is your phone number publicly visible?|Phone number reported as publicly visible|Limit phone-number visibility to only you.
email_public|personal|bad|2|Is your personal email publicly visible?|Personal email reported as public|Hide your email or use a masked contact address.
birthday_public|personal|bad|2|Is your full birth date visible?|Full birth date is public|Show only day and month to friends, or hide it.
workplace_public|personal|bad|2|Is your workplace publicly visible?|Workplace is publicly exposed|Hide employer details or limit them to connections.
education_public|personal|bad|1|Is your school or college publicly visible?|Education details are public|Limit education details to connections.
relationship_public|personal|bad|2|Are your family or relationship details public?|Family/relationship information is public|Restrict family and relationship information.
live_location|location|bad|3|Do you share your real-time location publicly?|Real-time location sharing enabled|Avoid broadcasting real-time location; share later or privately.
geotag|location|bad|2|Do your posts include geotags or check-ins?|Geotags/check-ins are used|Disable geotagging and public check-ins.
travel_posts|location|bad|2|Do you post travel plans before or during trips?|Travel plans are posted publicly|Post trip photos after you return, to a limited audience.
home_work_loc|location|bad|3|Do your posts reveal your home or workplace location?|Home/work location can be inferred|Avoid posting images or text that reveal your home or office.
posts_public|content|vis|3|Who can see your posts?|Posts are publicly visible|Set your default post audience to Friends.
photo_exif|content|bad|2|Do you upload photos without removing metadata (EXIF)?|Photo metadata is not managed|Strip EXIF/location metadata before uploading photos.
sensitive_photo|content|bad|3|Have you posted photos showing badges, documents, number plates or screens?|Photos may reveal badges, documents or screens|Review and remove photos that show IDs, badges, plates or screens.
kids_photos|content|bad|3|Do public photos show children or family members?|Family/children photos are public|Share family photos only with a small trusted audience.
unknown_accept|connections|bad|3|Do you accept connection requests from people you don't know?|Unknown connection requests are accepted|Verify unfamiliar profiles before accepting.
msg_open|connections|bad|2|Can anyone send you direct messages?|Direct messages are open to everyone|Restrict direct messages to people you know.
follower_approval|connections|good|2|Do you manually approve new followers?|Followers are not manually approved|Turn on manual follower approval.
connections_reviewed|connections|good|1|Do you regularly clean up your friends/follower list?|Connections are never reviewed|Review and remove unknown connections periodically.
anyone_tag|tagging|bad|2|Can anyone tag you without approval?|Anyone can tag you|Limit tagging to friends.
tag_review|tagging|good|3|Is tag review enabled (you approve tags first)?|Tag review is disabled|Enable tag review so you approve tags before they appear.
tag_auto_timeline|tagging|bad|2|Do tagged posts appear on your profile automatically?|Tagged posts appear automatically|Require approval for tagged posts on your profile.
mentions_open|tagging|bad|1|Can unknown users mention you?|Mentions are open to unknown users|Restrict mentions to people you follow.
mfa|account|good|3|Do you use multi-factor authentication?|MFA is disabled|Enable MFA using the strongest supported method (authenticator app or security key).
pw_unique|account|good|3|Do you use a unique password for this account?|Password reuse reported|Use a unique password; consider a password manager.
login_alerts|account|good|2|Are login alerts enabled?|Login alerts are disabled|Enable login alerts.
sessions_reviewed|account|good|1|Do you review active sessions and logged-in devices?|Active sessions are not reviewed|Review sessions and sign out of unknown devices.
apps_reviewed|apps|good|3|Do you review your connected third-party apps?|Third-party apps are never reviewed|Review connected apps and revoke what you no longer need.
unused_apps|apps|bad|2|Do you have old or unused apps still connected?|Unused apps retain access|Remove unused integrations (least privilege).
social_login|apps|bad|1|Do you use "Sign in with social account" on many sites?|Wide use of social sign-in|Prefer separate logins with unique passwords for sensitive services.
app_permissions|apps|good|2|Do you check the permissions before authorizing an app?|App permissions are not checked|Grant only the permissions an app truly needs.
suspicious_links|social|bad|3|Do you click unexpected links in messages?|Clicks unexpected links|Do not click unexpected links; verify through another channel.
share_codes|social|bad|3|Have you ever shared a verification code with someone?|Verification codes have been shared|Never share verification codes with anyone.
verify_impostors|social|good|2|Do you verify unfamiliar or look-alike accounts before replying?|Impersonation checks are not performed|Verify identity before responding to unfamiliar accounts.
giveaways|social|bad|1|Do you take part in giveaways or "free offer" posts?|Takes part in suspicious giveaways|Avoid offers that ask for personal details or links.
old_posts_reviewed|footprint|good|2|Do you regularly review old public posts?|Old posts are not reviewed|Audit old posts; delete or restrict the ones you no longer want public.
unused_accounts|footprint|bad|2|Do you have old, unused accounts that are still active?|Dormant accounts remain active|Close accounts you no longer use.
privacy_reviewed|footprint|good|2|Do you review your privacy settings at least twice a year?|Privacy settings are not reviewed|Schedule a privacy check-up every six months.
public_comments|footprint|bad|1|Do you leave public comments that contain personal details?|Personal details appear in public comments|Avoid personal details in public comments."""
OPTS = {"vis": ["PUBLIC", "FRIENDS", "PRIVATE"], "bad": ["YES", "SOMETIMES", "NOT SURE", "NO"], "good": ["YES", "SOMETIMES", "NOT SURE", "NO"]}
FACT = {"vis": {"PUBLIC": 1, "FRIENDS": .4, "PRIVATE": 0}, "bad": {"YES": 1, "SOMETIMES": .6, "NOT SURE": .5, "NO": 0},
        "good": {"YES": 0, "SOMETIMES": .5, "NOT SURE": .7, "NO": 1}}          # answer -> risk factor 0..1
BEST = {"vis": "PRIVATE", "bad": "NO", "good": "YES"}; WORST = {"vis": "PUBLIC", "bad": "YES", "good": "NO"}
PRI = {3: "IMMEDIATE", 2: "IMPORTANT", 1: "GOOD PRACTICE"}; SEV = {3: "HIGH", 2: "MEDIUM", 1: "LOW"}
PORD = {"IMMEDIATE": 0, "IMPORTANT": 1, "GOOD PRACTICE": 2}
QS = []
for ln in RAW.strip().splitlines():
    k, c, kind, w, t, f, r = ln.split("|")
    QS.append(dict(key=k, cat=c, kind=kind, w=int(w), text=t, finding=f, rec=r, options=OPTS[kind]))
QD = {q["key"]: q for q in QS}
SAFE = {k: BEST[q["kind"]] for k, q in QD.items()}; WORSTA = {k: WORST[q["kind"]] for k, q in QD.items()}
DEMO = {**SAFE, "profile_public": "PUBLIC", "posts_public": "PUBLIC", "phone_public": "YES", "birthday_public": "YES",
        "live_location": "YES", "geotag": "YES", "travel_posts": "YES", "unknown_accept": "YES", "tag_review": "NO",
        "mfa": "NO", "login_alerts": "NO", "apps_reviewed": "NO", "old_posts_reviewed": "NO"}   # fictional demo profile
CHECK = ["Review profile visibility", "Hide unnecessary contact information", "Review birth-date visibility", "Review location sharing",
         "Avoid unnecessary real-time location posts", "Review tagging permissions", "Review followers/friends", "Verify unfamiliar requests",
         "Enable MFA", "Enable login alerts where available", "Review active sessions", "Review connected apps", "Remove unused integrations",
         "Review old public posts", "Review photo privacy and metadata", "Be cautious with unexpected links", "Never share verification codes",
         "Review privacy settings periodically"]

# ---------------------------------------------------------------- 2. ENGINE
def validate(a):
    """Strict allow-list validation. Returns (answers, error)."""
    if not isinstance(a, dict): return None, "answers must be an object"
    if any(k not in QD for k in a): return None, "unknown question key"
    miss = [k for k in QD if k not in a]
    if miss: return None, f"{len(miss)} question(s) unanswered"
    for k, v in a.items():
        if not isinstance(v, str) or v not in OPTS[QD[k]["kind"]]: return None, f"invalid answer for {k}"
    return a, None

def extract_privacy_features(a):
    return {k: FACT[QD[k]["kind"]][a[k]] for k in QD}

def category_scores(f):
    out = {}
    for c in CATS:
        qs = [q for q in QS if q["cat"] == c]
        out[c] = round(100 * sum(q["w"] * f[q["key"]] for q in qs) / sum(q["w"] for q in qs), 1)
    return out

def classify(s): return "LOW" if s <= 20 else "MODERATE" if s <= 40 else "HIGH" if s <= 70 else "CRITICAL"

def calculate_privacy_risk(a, weights=WEIGHTS):
    f = extract_privacy_features(a); cs = category_scores(f)
    s = round(sum(weights[c] * cs[c] for c in cs))
    return {"score": s, "level": classify(s), "categories": cs, "features": f}

def generate_privacy_findings(f):
    out = [dict(key=k, category=QD[k]["cat"], severity=SEV[QD[k]["w"]], text=QD[k]["finding"], rec=QD[k]["rec"],
                priority=PRI[QD[k]["w"]], impact=round(QD[k]["w"] * v, 2)) for k, v in f.items() if v >= .5]
    return sorted(out, key=lambda x: (PORD[x["priority"]], -x["impact"]))

def generate_recommendations(findings):
    return [dict(priority=x["priority"], finding=x["text"], recommendation=x["rec"]) for x in findings]

def assess(a):
    r = calculate_privacy_risk(a); r["findings"] = generate_privacy_findings(r.pop("features")); return r

def simulate_improvement(a, fix):
    b = calculate_privacy_risk(a); n = dict(a)
    for k in fix: n[k] = BEST[QD[k]["kind"]]
    x = calculate_privacy_risk(n)
    return dict(before=b["score"], after=x["score"], reduction=b["score"] - x["score"], level_before=b["level"],
                level_after=x["level"], categories_after=x["categories"], disclaimer="Framework simulation, not a guarantee of real-world safety.")

# ---------------------------------------------------------------- 3. SYNTHETIC DATASET (1,000 fictional profiles)
def synth(n=1000, seed=42):
    rng = np.random.default_rng(seed); rows = []
    for i in range(n):
        c = rng.beta(2, 2.5); a = {}
        for k, q in QD.items():
            t = float(np.clip(c + rng.normal(0, .28), 0, 1))
            a[k] = min(FACT[q["kind"]], key=lambda o: abs(FACT[q["kind"]][o] - t))
        r = calculate_privacy_risk(a)
        rows.append({"profile_id": f"SYN-{i+1:04d}", **a, "risk_score": r["score"], "risk_level": r["level"],
                     **{"cat_" + c: v for c, v in r["categories"].items()}})
    return pd.DataFrame(rows)

DF = synth(); DF.to_csv(f"{ROOT}/data/social_media_privacy_assessments.csv", index=False)

def build_stats(df):
    lv = df.risk_level.value_counts().to_dict()
    cats = {CATS[c]: round(float(df["cat_" + c].mean()), 1) for c in CATS}
    share = lambda k: float((df[k].map(FACT[QD[k]["kind"]]) >= .5).mean() * 100)
    weak = sorted(((QD[k]["finding"], round(share(k), 1)) for k in QD), key=lambda x: -x[1])[:8]
    ck = {"mfa": "MFA", "pw_unique": "Unique password", "login_alerts": "Login alerts", "sessions_reviewed": "Sessions reviewed",
          "tag_review": "Tag review", "apps_reviewed": "Apps reviewed"}
    ctrl = [(v, round(float((df[k] == "YES").mean() * 100), 1)) for k, v in ck.items()]
    foot = [(QD[k]["finding"], round(share(k), 1)) for k in ("old_posts_reviewed", "unused_accounts", "privacy_reviewed", "public_comments")]
    recs = df[list(QD)].to_dict("records")
    after = lambda sel: round(float(np.mean([simulate_improvement(r, [k for k in QD if sel(QD[k])])["after"] for r in recs])), 1)
    return dict(n=len(df), avg=round(float(df.risk_score.mean()), 1), levels={l: int(lv.get(l, 0)) for l in ("LOW", "MODERATE", "HIGH", "CRITICAL")},
                cats=cats, weak=weak, controls=ctrl, foot=foot, improve=[round(float(df.risk_score.mean()), 1), after(lambda q: q["w"] == 3), after(lambda q: True)],
                high_cats=sum(v > 40 for v in cats.values()), controls_pct=round(sum(v for _, v in ctrl) / len(ctrl), 1), n_recs=len(QD))
STATS = build_stats(DF)

# ---------------------------------------------------------------- 4. PRIVACY-FIRST DATABASE (scores + finding types only)
def db():
    c = sqlite3.connect(DB); c.row_factory = sqlite3.Row; return c

def init_db():
    with db() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS assessments(assessment_id TEXT PRIMARY KEY, overall_score INTEGER, risk_level TEXT, created_at TEXT);
        CREATE TABLE IF NOT EXISTS category_scores(category_score_id INTEGER PRIMARY KEY AUTOINCREMENT, assessment_id TEXT, category TEXT, score REAL);
        CREATE TABLE IF NOT EXISTS findings(finding_id INTEGER PRIMARY KEY AUTOINCREMENT, assessment_id TEXT, category TEXT, finding_type TEXT, severity TEXT, description TEXT);
        CREATE TABLE IF NOT EXISTS recommendations(recommendation_id INTEGER PRIMARY KEY AUTOINCREMENT, finding_type TEXT UNIQUE, recommendation TEXT, priority TEXT);""")
        c.executemany("INSERT OR IGNORE INTO recommendations(finding_type,recommendation,priority) VALUES(?,?,?)",
                      [(k, q["rec"], PRI[q["w"]]) for k, q in QD.items()])
init_db()

def save(res):
    aid = "PRA-" + uuid.uuid4().hex[:8].upper(); now = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    with db() as c:
        c.execute("INSERT INTO assessments VALUES(?,?,?,?)", (aid, res["score"], res["level"], now))
        c.executemany("INSERT INTO category_scores(assessment_id,category,score) VALUES(?,?,?)", [(aid, k, v) for k, v in res["categories"].items()])
        c.executemany("INSERT INTO findings(assessment_id,category,finding_type,severity,description) VALUES(?,?,?,?,?)",
                      [(aid, f["category"], f["key"], f["severity"], f["text"]) for f in res["findings"]])
    return aid

def load(aid):
    if not re.fullmatch(r"PRA-[A-F0-9]{8}", aid): return None
    with db() as c:
        a = c.execute("SELECT * FROM assessments WHERE assessment_id=?", (aid,)).fetchone()
        if not a: return None
        cs = {r["category"]: r["score"] for r in c.execute("SELECT category,score FROM category_scores WHERE assessment_id=?", (aid,))}
        fs = [dict(r) for r in c.execute("SELECT f.category,f.finding_type,f.severity,f.description,r.recommendation,r.priority FROM findings f "
                                         "LEFT JOIN recommendations r ON r.finding_type=f.finding_type WHERE f.assessment_id=?", (aid,))]
    return dict(id=aid, score=a["overall_score"], level=a["risk_level"], created_at=a["created_at"], categories=cs,
                findings=sorted(fs, key=lambda x: PORD.get(x["priority"], 9)))

# ---------------------------------------------------------------- 5. REPORT (HTML, print to PDF from the browser)
LC = {"LOW": "#059669", "MODERATE": "#B45309", "HIGH": "#C2410C", "CRITICAL": "#BE123C"}
def report_html(r):
    e = html.escape
    cat = "".join(f"<tr><td>{e(CATS.get(c, c))}</td><td>{e(str(s))}/100</td></tr>" for c, s in r["categories"].items())
    fnd = "".join(f"<li><b>[{e(str(f.get('priority') or '-'))}]</b> {e(str(f['description']))}<br><i>Action:</i> {e(str(f.get('recommendation') or ''))}</li>"
                  for f in r["findings"]) or "<li>No significant findings.</li>"
    chk = "".join(f"<li>&#9744; {e(x)}</li>" for x in CHECK)
    css = ("body{font:15px/1.6 system-ui,sans-serif;max-width:820px;margin:30px auto;padding:0 20px;color:#111}h1{color:#312E81}"
           "table{border-collapse:collapse;width:100%}td{border:1px solid #ccc;padding:6px 10px}.lv{color:#fff;padding:4px 14px;border-radius:99px;font-weight:700}"
           "li{margin:6px 0}small{color:#555}@media print{button{display:none}}")
    return (f"<!doctype html><meta charset=utf-8><title>Privacy Report {e(r['id'])}</title><style>{css}</style>"
            f"<button onclick='print()'>Print / Save as PDF</button><h1>Social Media Privacy Risk Assessment Report</h1>"
            f"<p><b>Assessment ID:</b> {e(r['id'])}<br><b>Date:</b> {e(str(r['created_at']))}<br><b>Overall risk:</b> {e(str(r['score']))}/100 "
            f"<span class=lv style='background:{LC.get(r['level'], '#333')}'>{e(r['level'])}</span></p>"
            f"<h2>Category scores (higher = more risk)</h2><table>{cat}</table><h2>Findings and priority actions</h2><ol>{fnd}</ol>"
            f"<h2>Privacy checklist</h2><ul style='list-style:none;padding:0'>{chk}</ul>"
            f"<p><small><b>Disclaimer.</b> {e(DISCLAIMER)} This report holds scores and finding types only; no personal data was collected.</small></p>")

# ---------------------------------------------------------------- 6. DASHBOARD (HTML / CSS / JS + Chart.js)
PAGE = r"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>PrivacyShield | Social Media Privacy Risk Dashboard</title>
<link href="https://fonts.googleapis.com/css2?family=Manrope:wght@400;500;700;800&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.3/dist/chart.umd.min.js"></script>
<style>
*{box-sizing:border-box;margin:0}
:root{--bg:#090E1D;--card:#111A33;--card2:#16213F;--bd:rgba(160,180,230,.2);--tx:#F1F4FF;--mu:#AEBAD8;--ac:#3EE6C3;--vi:#8B7BFF}
body{font:15px/1.55 Manrope,system-ui,sans-serif;color:var(--tx);min-height:100vh;
 background:radial-gradient(900px 520px at 8% -8%,rgba(139,123,255,.26),transparent),radial-gradient(800px 480px at 100% 105%,rgba(62,230,195,.14),transparent),var(--bg)}
button{font-family:inherit}
.top{display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:12px;padding:14px 5vw;position:sticky;top:0;z-index:9;background:rgba(9,14,29,.86);backdrop-filter:blur(14px);border-bottom:1px solid var(--bd)}
.brand{display:flex;gap:12px;align-items:center}.brand b{font-size:18px;display:block;line-height:1.2}.brand small{color:var(--mu)}
.logo{width:42px;height:42px;display:grid;place-items:center;border-radius:12px;background:linear-gradient(135deg,var(--vi),var(--ac));font-size:21px}
nav{display:flex;gap:4px;background:var(--card);padding:4px;border-radius:12px;border:1px solid var(--bd)}
nav button{background:none;border:0;color:var(--mu);padding:8px 16px;border-radius:9px;font-weight:700;font-size:14px;cursor:pointer}
nav button.on{background:var(--vi);color:#fff}
main{max-width:1240px;margin:auto;padding:26px 5vw 50px}section[hidden]{display:none}
.hero{padding:34px;border-radius:22px;border:1px solid var(--bd);background:linear-gradient(135deg,rgba(139,123,255,.24),rgba(62,230,195,.08));margin-bottom:20px}
h1{font-size:clamp(26px,4vw,42px);letter-spacing:-.025em;line-height:1.12;font-weight:800;max-width:760px}
.hero p{color:var(--mu);max-width:700px;margin:12px 0 22px}
.btn{display:inline-block;text-decoration:none;border:0;cursor:pointer;color:#06231D;font-weight:800;font-size:15px;padding:13px 24px;border-radius:10px;background:var(--ac);box-shadow:0 8px 24px rgba(62,230,195,.25)}
.btn.ghost{background:var(--card2);color:var(--tx);border:1px solid var(--bd);box-shadow:none}
.grid{display:grid;gap:18px}.kpis{grid-template-columns:repeat(auto-fit,minmax(190px,1fr));margin-bottom:18px}.g3{grid-template-columns:repeat(auto-fit,minmax(340px,1fr))}
.card{background:var(--card);border:1px solid var(--bd);border-radius:16px;padding:20px}
.card h3{font-size:15px;margin-bottom:14px;font-weight:800}
.k small{color:var(--mu);font-weight:700;font-size:13px}.k b{display:block;font-size:30px;margin-top:2px;font-weight:800}
.cv{position:relative;height:300px}.mu{color:var(--mu)}.c{text-align:center;margin-top:12px}
.q{display:flex;justify-content:space-between;gap:14px;align-items:center;flex-wrap:wrap;padding:11px 0;border-top:1px solid var(--bd)}
.q span{flex:1 1 330px}.seg{display:flex;gap:4px;flex-wrap:wrap}
.seg button{background:var(--card2);border:1px solid var(--bd);color:var(--tx);padding:6px 12px;border-radius:8px;font-size:12px;font-weight:700;cursor:pointer}
.seg button.on{background:var(--vi);border-color:#B5ABFF;color:#fff}
.pill{display:inline-block;padding:3px 11px;border-radius:99px;font-size:12px;font-weight:800;color:#0A0F1E}
.ring{width:200px;height:200px;border-radius:50%;display:grid;place-items:center;margin:auto}
.ring>div{width:154px;height:154px;border-radius:50%;background:var(--card);display:grid;place-items:center;text-align:center}
.ring b{font-size:46px;line-height:1;font-weight:800;display:block}
.f{padding:12px 0;border-top:1px solid var(--bd)}.f b{font-weight:700}.fx2{display:block;padding:7px 0;cursor:pointer}
input[type=checkbox]{accent-color:#8B7BFF;width:16px;height:16px;vertical-align:-3px}
.so{margin-top:16px;font-size:20px}.so b{font-size:36px;font-weight:800}.s{font-size:12px;margin-top:6px}
.pg{height:8px;border-radius:9px;background:rgba(255,255,255,.12);flex:1;min-width:160px}.pg i{display:block;height:100%;border-radius:9px;background:linear-gradient(90deg,var(--vi),var(--ac));width:0;transition:width .3s}
.bar{display:flex;gap:14px;align-items:center;flex-wrap:wrap;margin-bottom:16px}.row{display:flex;gap:12px;flex-wrap:wrap;margin-top:18px}
.btn:disabled{opacity:.45;cursor:not-allowed}footer{padding:0 5vw 40px;max-width:1240px;margin:auto;color:var(--mu);font-size:13px}
@media print{body{background:#fff;color:#000}.top,.hero,.row,footer,.noprint{display:none}.card{border-color:#999;background:#fff}}
</style></head><body>
<header class="top"><div class="brand"><div class="logo">&#128737;</div><div><b>PrivacyShield</b><small>Social media privacy risk framework</small></div></div>
<nav><button data-t="dash" class="on">Dashboard</button><button data-t="test">Assessment</button><button data-t="res">Results</button><button data-t="chk">Checklist</button></nav></header>
<main>
<section id="dash"><div class="hero"><h1>See how exposed your social profile is, and what to fix first.</h1>
<p>Answer 42 short questions about your settings and habits. You get a 0 to 100 risk score, category breakdown, and prioritized fixes. No passwords, phone numbers or addresses are ever asked for. The charts below use 1,000 synthetic profiles.</p>
<button class="btn" id="start">Start Privacy Assessment</button></div>
<div class="grid kpis" id="kpis"></div>
<div class="grid g3">
<div class="card"><h3>Category risk scores (average)</h3><div class="cv"><canvas id="c1"></canvas></div></div>
<div class="card"><h3>Privacy risk distribution</h3><div class="cv"><canvas id="c2"></canvas></div></div>
<div class="card"><h3>Top privacy weaknesses (% of profiles)</h3><div class="cv"><canvas id="c3"></canvas></div></div>
<div class="card"><h3>Account security controls enabled (%)</h3><div class="cv"><canvas id="c4"></canvas></div></div>
<div class="card"><h3>Digital footprint risk (% of profiles)</h3><div class="cv"><canvas id="c5"></canvas></div></div>
<div class="card"><h3>Privacy improvement comparison (average score)</h3><div class="cv"><canvas id="c6"></canvas></div></div></div></section>

<section id="test" hidden><div class="card" style="margin-bottom:16px"><h3>Privacy questionnaire</h3>
<div class="bar"><div class="pg"><i id="pgb"></i></div><span class="mu" id="pgl"></span>
<button class="btn ghost" id="demo">Load fictional demo profile</button><button class="btn ghost" id="rst">Reset</button></div>
<p class="mu">Only answer whether something is visible or enabled. Never type the information itself.</p></div>
<div id="qs"></div><button class="btn" id="go" disabled>Calculate my privacy risk</button></section>

<section id="res" hidden><div class="card"><h3>No results yet</h3><p class="mu">Complete the assessment to see your score.</p></div></section>

<section id="chk" hidden><div class="card"><h3>Social media privacy checklist</h3><div id="cl"></div>
<div class="row noprint"><button class="btn" onclick="print()">Print checklist</button><button class="btn ghost" id="dl">Download as text</button></div></div></section>
</main>
<footer id="dis"></footer>
<script>
const $=s=>document.querySelector(s);
const E=(t,c,x)=>{const e=document.createElement(t);if(c)e.className=c;if(x!==undefined)e.textContent=x;return e};
const COL={LOW:'#34D399',MODERATE:'#FBBF24',HIGH:'#FB923C',CRITICAL:'#F43F5E'},PC={'IMMEDIATE':'#F43F5E','IMPORTANT':'#FB923C','GOOD PRACTICE':'#3EE6C3'};
const lvl=s=>s<=20?'LOW':s<=40?'MODERATE':s<=70?'HIGH':'CRITICAL';
const api=async(u,o)=>{const r=await fetch(u,o),j=await r.json();if(!r.ok)throw Error(j.error||'Request failed');return j};
const post=(u,b)=>api(u,{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(b)});
Chart.defaults.color='#C9D4EE';Chart.defaults.borderColor='rgba(160,180,230,.16)';Chart.defaults.font.family='Manrope,system-ui,sans-serif';
let Q=[],CATS={},ans={},cur=null,CH={};
const mk=(id,cfg)=>{CH[id]&&CH[id].destroy();CH[id]=new Chart(document.getElementById(id),cfg)};
const show=t=>{document.querySelectorAll('main>section').forEach(s=>s.hidden=s.id!==t);document.querySelectorAll('nav button').forEach(b=>b.classList.toggle('on',b.dataset.t===t));scrollTo(0,0)};
document.querySelectorAll('[data-t]').forEach(b=>b.onclick=()=>show(b.dataset.t));$('#start').onclick=()=>show('test');
const bar=(id,l,v,col,h,max)=>mk(id,{type:'bar',data:{labels:l,datasets:[{data:v,backgroundColor:col,borderRadius:7,maxBarThickness:26}]},
 options:{indexAxis:h?'y':'x',maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{[h?'x':'y']:{beginAtZero:true,max:max}}}});

async function loadDash(){
 const s=await api('/api/dashboard/stats'),L=lvl(s.avg),k=$('#kpis');k.innerHTML='';
 [['Average risk score',s.avg+' / 100',COL[L]],['Population risk level',L,COL[L]],['High-risk categories',s.high_cats+' of 10','#FB923C'],
  ['Recommendations mapped',s.n_recs,'#8B7BFF'],['Security controls enabled',s.controls_pct+'%','#3EE6C3']].forEach(([a,b,c])=>{
  const d=E('div','card k'),v=E('b','',b);v.style.color=c;d.append(E('small','',a),v);k.append(d)});
 const cl=Object.keys(s.cats),cv=Object.values(s.cats);
 bar('c1',cl,cv,cv.map(v=>COL[lvl(v)]),true,100);
 mk('c2',{type:'doughnut',data:{labels:Object.keys(s.levels),datasets:[{data:Object.values(s.levels),backgroundColor:Object.keys(s.levels).map(l=>COL[l]),borderWidth:0}]},
  options:{maintainAspectRatio:false,cutout:'64%',plugins:{legend:{position:'bottom'}}}});
 bar('c3',s.weak.map(x=>x[0]),s.weak.map(x=>x[1]),'#8B7BFF',true,100);
 bar('c4',s.controls.map(x=>x[0]),s.controls.map(x=>x[1]),'#3EE6C3',false,100);
 bar('c5',s.foot.map(x=>x[0]),s.foot.map(x=>x[1]),'#FBBF24',true,100);
 bar('c6',['Current average','Fix top-priority items','Fix every finding'],s.improve,s.improve.map(v=>COL[lvl(v)]),false,100);
}
async function loadQ(){
 const d=await api('/api/questions');Q=d.questions;CATS=d.cats;$('#dis').textContent=d.disclaimer;const box=$('#qs');box.innerHTML='';
 Object.keys(CATS).forEach((c,i)=>{const card=E('div','card');card.style.marginBottom='16px';card.append(E('h3','',String.fromCharCode(65+i)+'. '+CATS[c]));
  Q.filter(q=>q.cat===c).forEach(q=>{const row=E('div','q'),seg=E('div','seg');row.append(E('span','',q.text),seg);
   q.options.forEach(o=>{const b=E('button','',o);b.dataset.k=q.key;b.dataset.o=o;
    b.onclick=()=>{ans[q.key]=o;seg.querySelectorAll('button').forEach(x=>x.classList.toggle('on',x===b));prog()};seg.append(b)});card.append(row)});box.append(card)});prog();
}
function setAns(a){ans={...a};document.querySelectorAll('.seg button').forEach(b=>b.classList.toggle('on',ans[b.dataset.k]===b.dataset.o));prog()}
function prog(){const n=Object.keys(ans).length;$('#pgl').textContent=n+' of '+Q.length+' answered';$('#pgb').style.width=(100*n/Q.length)+'%';$('#go').disabled=n<Q.length}
$('#demo').onclick=async()=>setAns((await api('/api/demo')).answers);$('#rst').onclick=()=>setAns({});
$('#go').onclick=async()=>{try{cur=await post('/api/assessment',{answers:ans});cur.ans={...ans};renderRes();show('res')}catch(e){alert(e.message)}};

function renderRes(){
 const r=cur,R=$('#res'),c=COL[r.level];
 R.innerHTML=`<div class="grid g3"><div class="card"><h3>Overall privacy risk</h3><div class="ring" id="ring"><div><span><b id="sc"></b><small class="mu">out of 100</small></span></div></div>
 <p class="c"><span class="pill" id="lv"></span></p><p class="mu c" id="rid"></p></div>
 <div class="card"><h3>Category risk radar (higher = more risk)</h3><div class="cv"><canvas id="rad"></canvas></div></div></div>
 <div class="grid g3" style="margin-top:18px"><div class="card"><h3>Findings and recommendations</h3><div id="fl"></div></div>
 <div class="card"><h3>Privacy improvement simulator</h3><p class="mu">Tick the fixes you would apply. The score recalculates instantly.</p><div id="sf" style="margin-top:8px"></div><div class="so" id="so"></div></div></div>
 <div class="row"><a class="btn" id="rep" target="_blank">Open printable report</a><button class="btn ghost" id="del">Delete this assessment</button></div>`;
 $('#sc').textContent=r.score;$('#sc').style.color=c;$('#lv').textContent=r.level;$('#lv').style.background=c;$('#rid').textContent='Assessment ID: '+r.id;
 $('#ring').style.background=`conic-gradient(${c} ${r.score*3.6}deg,rgba(255,255,255,.12) 0)`;$('#rep').href='/report/'+r.id;
 $('#del').onclick=async()=>{await api('/api/assessment/'+r.id,{method:'DELETE'});cur=null;R.innerHTML='<div class="card"><h3>Assessment deleted</h3><p class="mu">Your stored scores were removed.</p></div>'};
 if(!r.findings.length)$('#fl').append(E('p','mu','No significant findings. Keep reviewing your settings regularly.'));
 r.findings.forEach(f=>{const d=E('div','f'),p=E('span','pill',f.priority);p.style.background=PC[f.priority];d.append(p,E('b','',' '+f.text),E('div','mu',f.rec));$('#fl').append(d);
  const l=E('label','fx2'),cb=E('input');cb.type='checkbox';cb.className='fx';cb.value=f.key;cb.onchange=sim;l.append(cb,document.createTextNode(' '+f.text));$('#sf').append(l)});
 const ks=Object.keys(r.categories);
 mk('rad',{type:'radar',data:{labels:ks.map(k=>CATS[k]),datasets:[
  {label:'Current',data:ks.map(k=>r.categories[k]),borderColor:'#F43F5E',backgroundColor:'rgba(244,63,94,.24)',pointBackgroundColor:'#F43F5E'},
  {label:'Simulated',data:ks.map(()=>null),borderColor:'#3EE6C3',backgroundColor:'rgba(62,230,195,.22)',pointBackgroundColor:'#3EE6C3'}]},
  options:{maintainAspectRatio:false,plugins:{legend:{position:'bottom'}},scales:{r:{min:0,max:100,ticks:{stepSize:25,backdropColor:'transparent',color:'#AEBAD8'},
  grid:{color:'rgba(160,180,230,.22)'},angleLines:{color:'rgba(160,180,230,.22)'},pointLabels:{color:'#F1F4FF',font:{size:11}}}}}});
}
async function sim(){
 const fix=[...document.querySelectorAll('.fx:checked')].map(x=>x.value),o=$('#so'),ks=Object.keys(cur.categories);o.innerHTML='';
 if(!fix.length){CH.rad.data.datasets[1].data=ks.map(()=>null);CH.rad.update();return}
 const s=await post('/api/assessment/simulate-improvement',{answers:cur.ans,fix}),a=E('b','',s.before),b=E('b','',s.after);
 a.style.color=COL[s.level_before];b.style.color=COL[s.level_after];
 o.append(a,E('span','',' \u2192 '),b,E('span','mu','   \u2212'+s.reduction+' points, now '+s.level_after),E('div','mu s',s.disclaimer));
 CH.rad.data.datasets[1].data=ks.map(k=>s.categories_after[k]);CH.rad.update();
}
async function loadChk(){const d=await api('/api/privacy-checklist');d.items.forEach(t=>{const l=E('label','fx2'),c=E('input');c.type='checkbox';l.append(c,document.createTextNode(' '+t));$('#cl').append(l)})}
$('#dl').onclick=()=>{const t=[...document.querySelectorAll('#cl label')].map(l=>'[ ] '+l.textContent.trim()).join('\n'),a=E('a');
 a.href=URL.createObjectURL(new Blob(['SOCIAL MEDIA PRIVACY CHECKLIST\n\n'+t],{type:'text/plain'}));a.download='privacy_checklist.txt';a.click()};
loadDash().catch(console.error);loadQ().catch(console.error);loadChk().catch(console.error);
</script></body></html>"""

# ---------------------------------------------------------------- 7. REST API (Flask)
logging.getLogger("werkzeug").setLevel(logging.ERROR)
app = Flask(__name__); HITS = {}; RATE = 600     # requests / minute / IP (simple in-memory rate limit)

@app.before_request
def guard():
    ip, now = request.remote_addr, time.time()
    h = [t for t in HITS.get(ip, []) if now - t < 60] + [now]; HITS[ip] = h
    if len(h) > RATE: return jsonify(error="Rate limit exceeded"), 429
    if request.content_length and request.content_length > 20000: return jsonify(error="Payload too large"), 413

@app.after_request
def secure(r):
    r.headers.update({"X-Content-Type-Options": "nosniff", "Referrer-Policy": "no-referrer", "Cache-Control": "no-store"}); return r

@app.get("/")
def home(): return Response(PAGE, mimetype="text/html")

@app.get("/api/questions")
def api_questions():
    return jsonify(cats=CATS, disclaimer=DISCLAIMER, questions=[{k: q[k] for k in ("key", "cat", "text", "options")} for q in QS])

@app.get("/api/demo")
def api_demo(): return jsonify(answers=DEMO)

@app.post("/api/assessment")
def api_new():
    a, err = validate((request.get_json(silent=True) or {}).get("answers"))
    if err: return jsonify(error=err), 400
    res = assess(a); res["id"] = save(res); res["disclaimer"] = DISCLAIMER   # answers themselves are never stored
    return jsonify(res), 201

@app.get("/api/assessment/<aid>")
def api_get(aid):
    r = load(aid); return (jsonify(r), 200) if r else (jsonify(error="Not found"), 404)

@app.get("/api/assessment/<aid>/recommendations")
def api_recs(aid):
    r = load(aid)
    if not r: return jsonify(error="Not found"), 404
    return jsonify(recommendations=[dict(priority=f["priority"], finding=f["description"], recommendation=f["recommendation"]) for f in r["findings"]])

@app.post("/api/assessment/simulate-improvement")
def api_sim():
    d = request.get_json(silent=True) or {}; a, err = validate(d.get("answers")); fix = d.get("fix")
    if err: return jsonify(error=err), 400
    if not isinstance(fix, list) or any(k not in QD for k in fix): return jsonify(error="invalid fix list"), 400
    return jsonify(simulate_improvement(a, fix))

@app.get("/api/dashboard/stats")
def api_stats():
    with db() as c: live = c.execute("SELECT COUNT(*) FROM assessments").fetchone()[0]
    return jsonify({**STATS, "live": live})

@app.get("/api/privacy-checklist")
def api_check(): return jsonify(items=CHECK)

@app.delete("/api/assessment/<aid>")
def api_del(aid):
    if not load(aid): return jsonify(error="Not found"), 404
    with db() as c:
        for t in ("category_scores", "findings", "assessments"): c.execute(f"DELETE FROM {t} WHERE assessment_id=?", (aid,))
    return jsonify(deleted=True)

@app.get("/report/<aid>")
def report(aid):
    r = load(aid); return (Response(report_html(r), mimetype="text/html"), 200) if r else (Response("Not found", status=404))

# ---------------------------------------------------------------- 8. AUTOMATED TESTS (37)
def run_tests():
    global DB
    real = DB; DB = f"{ROOT}/tests/test.db"
    if os.path.exists(DB): os.remove(DB)
    init_db(); cl = app.test_client(); T = []
    W = lambda ks: {**SAFE, **{k: WORSTA[k] for k in ks}}
    def add(s, e, f): T.append((f"T{len(T)+1:02d}", s, e, f))
    r0, r1 = calculate_privacy_risk(SAFE), calculate_privacy_risk(WORSTA)
    add("Fully private profile", "0 / LOW", lambda: (r0["score"] == 0 and r0["level"] == "LOW", f'{r0["score"]} / {r0["level"]}'))
    add("Fully public profile", "100 / CRITICAL", lambda: (r1["score"] == 100 and r1["level"] == "CRITICAL", f'{r1["score"]} / {r1["level"]}'))
    SC = {"phone_public": "Public phone", "email_public": "Public email", "birthday_public": "Public birthday", "live_location": "Public location sharing",
          "geotag": "Real-time check-ins / geotags", "travel_posts": "Travel plans posted", "workplace_public": "Workplace exposure", "education_public": "Education exposure",
          "posts_public": "Public posts", "unknown_accept": "Unknown connections accepted", "tag_review": "Tag review disabled", "mfa": "MFA disabled",
          "login_alerts": "Login alerts disabled", "pw_unique": "Password reuse reported", "apps_reviewed": "Third-party apps not reviewed",
          "suspicious_links": "Low suspicious-link awareness", "old_posts_reviewed": "Old posts not reviewed", "privacy_reviewed": "Privacy settings not reviewed"}
    for k, name in SC.items():
        def fn(k=k):
            r = assess(W([k])); return (r["score"] > 0 and any(f["key"] == k for f in r["findings"]), f'score {r["score"]}, finding raised')
        add(name, "score > 0 and finding raised", fn)
    ak = [k for k, q in QD.items() if q["cat"] == "account"]
    def t_cat():
        c = calculate_privacy_risk(W(ak))["categories"]; return (c["account"] == 100 and sum(v for k, v in c.items() if k != "account") == 0, f'account={c["account"]}')
    add("Category score calculation", "account=100, others 0", t_cat)
    exp = round(100 * WEIGHTS["account"])
    add("Overall score calculation (weights)", f"score {exp}", lambda: (calculate_privacy_risk(W(ak))["score"] == exp, str(calculate_privacy_risk(W(ak))["score"])))
    for lo, hi, a, b in [(20, 21, "LOW", "MODERATE"), (40, 41, "MODERATE", "HIGH"), (70, 71, "HIGH", "CRITICAL")]:
        add(f"Score boundary {lo}/{hi}", f"{lo}={a}, {hi}={b}", lambda lo=lo, hi=hi, a=a, b=b: (classify(lo) == a and classify(hi) == b, f"{classify(lo)}, {classify(hi)}"))
    def t_rec():
        rc = generate_recommendations(assess(WORSTA)["findings"]); return (len(rc) >= 30 and rc[0]["priority"] == "IMMEDIATE", f'{len(rc)} recs, first={rc[0]["priority"]}')
    add("Recommendation generation", ">=30 recs, IMMEDIATE first", t_rec)
    def t_sim():
        s = simulate_improvement(DEMO, ["mfa", "phone_public", "live_location", "tag_review"]); return (s["after"] < s["before"], f'{s["before"]} -> {s["after"]}')
    add("Improvement simulation", "after < before", t_sim)
    def t_db():
        r = cl.post("/api/assessment", json={"answers": DEMO}); i = r.get_json()["id"]; g = cl.get(f"/api/assessment/{i}")
        return (r.status_code == 201 and g.status_code == 200 and g.get_json()["score"] == calculate_privacy_risk(DEMO)["score"], f"{r.status_code}/{g.status_code} {i}")
    add("Database save and fetch", "201 then 200, same score", t_db)
    def t_sens():
        T4 = ("assessments", "category_scores", "findings", "recommendations")
        with sqlite3.connect(DB) as c:
            cols = [r[1] for t in T4 for r in c.execute(f"PRAGMA table_info({t})")]
            txt = " | ".join(v for t in T4 for row in c.execute(f"SELECT * FROM {t}") for v in row if isinstance(v, str))
        bad = [x for x in cols if re.search("phone|email|address|birth|password|location|message|answer", x)]
        pii = re.search(r"[\w.]+@[\w.]+\.\w+|(\+?\d[\s-]?){10,}", txt)
        return (not bad and not pii, f"sensitive columns={bad}, PII pattern={bool(pii)}")
    add("Sensitive data not stored", "no sensitive columns / PII", t_sens)
    def t_rep():
        i = cl.post("/api/assessment", json={"answers": DEMO}).get_json()["id"]; h = cl.get(f"/report/{i}"); b = h.get_data(as_text=True)
        return (h.status_code == 200 and i in b and "Disclaimer" in b, f"HTTP {h.status_code}")
    add("Report generation", "200 with ID and disclaimer", t_rep)
    def t_val():
        b1 = {k: v for k, v in DEMO.items() if k != "mfa"}; b2 = {**DEMO, "mfa": "MAYBE"}; b3 = {**DEMO, "<script>alert(1)</script>": "YES"}
        codes = [cl.post("/api/assessment", json={"answers": b}).status_code for b in (b1, b2, b3)]; return (codes == [400, 400, 400], str(codes))
    add("Input validation (missing, invalid, unknown key)", "[400, 400, 400]", t_val)
    def t_xss():
        h = report_html(dict(id="PRA-00000000", score=1, level="LOW", created_at="x", categories={"profile": 1}, findings=[dict(description="<script>alert(1)</script>", recommendation="<img src=x onerror=1>", priority="IMPORTANT")]))
        return ("<script>alert" not in h and "<img src=x" not in h, "tags escaped")
    add("XSS protection in report", "HTML is escaped", t_xss)
    def t_del():
        i = cl.post("/api/assessment", json={"answers": DEMO}).get_json()["id"]; d = cl.delete(f"/api/assessment/{i}").status_code; g = cl.get(f"/api/assessment/{i}").status_code
        return (d == 200 and g == 404, f"delete={d}, then get={g}")
    add("Data deletion", "delete 200, get 404", t_del)
    add("Weights sum to 1.0", "1.0", lambda: (abs(sum(WEIGHTS.values()) - 1) < 1e-9, str(round(sum(WEIGHTS.values()), 6))))
    add("Questionnaire size", ">= 40 questions", lambda: (len(QS) >= 40, f"{len(QS)} questions"))
    add("Synthetic dataset", "1000 rows, no PII columns", lambda: (len(DF) >= 1000 and not [c for c in DF.columns if re.search("phone_number|email_address|address|password", c)], f"{len(DF)} rows"))
    add("Security headers", "nosniff present", lambda: (cl.get("/").headers.get("X-Content-Type-Options") == "nosniff", "header present"))
    rows = []
    for i, s, e, f in T:
        try: ok, a = f()
        except Exception as x: ok, a = False, repr(x)
        rows.append((i, s, e, a, "PASS" if ok else "FAIL"))
    DB = real; HITS.clear()
    return pd.DataFrame(rows, columns=["Test ID", "Scenario", "Expected", "Actual", "Result"])

res = run_tests(); res.to_csv(f"{ROOT}/tests/test_results.csv", index=False)
print(res.to_string(index=False, max_colwidth=52))
print(f"\n{(res.Result == 'PASS').sum()} / {len(res)} tests passed\n")

# ---------------------------------------------------------------- 9. LAUNCH SERVER + PUBLIC URL
def free_port():
    s = socket.socket(); s.bind(("", 0)); p = s.getsockname()[1]; s.close(); return p
PORT = free_port()
threading.Thread(target=lambda: app.run(host="127.0.0.1", port=PORT, debug=False, use_reloader=False), daemon=True).start(); time.sleep(1.5)

def tunnel(port):
    """Free public HTTPS link via a Cloudflare quick tunnel (no account needed)."""
    try:
        subprocess.run(["pkill", "-x", "cloudflared"])
        if not os.access("/tmp/cloudflared", os.X_OK):
            subprocess.run("wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared", shell=True, timeout=90)
        if not os.access("/tmp/cloudflared", os.X_OK): return None
        log = open("/tmp/cf.log", "w")
        subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"], stdout=log, stderr=log)
        for _ in range(45):
            time.sleep(1); m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/tmp/cf.log").read())
            if m: return m.group(0)
    except Exception: pass
    return None

URL = tunnel(PORT); PROXY = None
try:
    from google.colab import output
    PROXY = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
except Exception: pass

print("=" * 70)
print("  DASHBOARD (public link, works in any browser):", URL or "tunnel unavailable")
print("  Colab-only fallback link (needs your Google login):", PROXY or "n/a")
print(f"  Local: http://127.0.0.1:{PORT}   |   Files: {ROOT}")
print("=" * 70)
try:
    from IPython.display import display, HTML
    link = URL or PROXY
    if link:
        display(HTML(f'<div style="font-family:system-ui;padding:18px 22px;border-radius:14px;background:#111A33;border:1px solid #3EE6C3;max-width:640px">'
                     f'<div style="color:#AEBAD8;font-size:13px;margin-bottom:8px">PrivacyShield dashboard is live</div>'
                     f'<a href="{link}" target="_blank" style="color:#06231D;background:#3EE6C3;font-weight:800;padding:11px 22px;border-radius:10px;text-decoration:none;display:inline-block">Open dashboard in new tab</a>'
                     f'<div style="color:#F1F4FF;font-size:13px;margin-top:10px;word-break:break-all">{link}</div></div>'))
    if PROXY:
        output.serve_kernel_port_as_iframe(PORT, height=820)   # inline preview inside Colab
except Exception: pass

Test ID                                         Scenario                     Expected                                  Actual Result
    T01                            Fully private profile                      0 / LOW                                 0 / LOW   PASS
    T02                             Fully public profile               100 / CRITICAL                          100 / CRITICAL   PASS
    T03                                     Public phone score > 0 and finding raised                 score 4, finding raised   PASS
    T04                                     Public email score > 0 and finding raised                 score 3, finding raised   PASS
    T05                                  Public birthday score > 0 and finding raised                 score 3, finding raised   PASS
    T06                          Public location sharing score > 0 and finding raised                 score 4, finding raised   PASS
    T07                    Real-time check-ins / geotags score > 0 an

<IPython.core.display.Javascript object>